In [ ]:
import os, glob, random, json

DATA_DIR   = '/kaggle/input/datasets/zphilip/nougat-training-dataset-example'
OUTPUT_DIR = '/kaggle/working/vlm_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Explore dataset structure
for root, dirs, files in os.walk(DATA_DIR):
    for f in files[:3]:
        print(os.path.join(root, f))

In [ ]:
images = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.png'), recursive=True))
mmds   = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.mmd'), recursive=True))
print('Total images:', len(images), ' | Total markdowns:', len(mmds))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for i, ax in enumerate(axes):
    img = Image.open(images[i]).convert('RGB')
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Document Sample {i+1}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'dataset_samples.png'), dpi=80)
plt.show()

# Show a markdown sample
with open(mmds[0]) as f:
    print(f.read()[:600])

In [ ]:
mmd_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in mmds}

pairs = []
for img_path in images:
    key = os.path.splitext(os.path.basename(img_path))[0]
    if key in mmd_lookup:
        pairs.append({'image': img_path, 'markdown': mmd_lookup[key]})

print('Valid image-markdown pairs:', len(pairs))

In [ ]:
random.seed(42)
random.shuffle(pairs)

split       = int(0.8 * len(pairs))
train_pairs = pairs[:split]
val_pairs   = pairs[split:]

with open(os.path.join(OUTPUT_DIR, 'train_pairs.json'), 'w') as f:
    json.dump(train_pairs, f)
with open(os.path.join(OUTPUT_DIR, 'val_pairs.json'), 'w') as f:
    json.dump(val_pairs, f)

print('Train:', len(train_pairs), ' | Val:', len(val_pairs))

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers>=4.45.0', 'peft>=0.12.0',
                'bitsandbytes>=0.43.0', 'trl>=0.11.0', 'accelerate>=0.34.0',
                'qwen-vl-utils', 'datasets'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Dual-GPU (T4 × 2) — GPU enforcement & environment setup ──────────────────
import os, sys, warnings

# MUST be set before any CUDA context is created
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch

# Hard-fail if no GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Kaggle → Settings → Accelerator → GPU T4 x2"
    )

import torch.autograd.graph as _tag
_tag.set_warn_on_accumulate_grad_stream_mismatch(False)
warnings.filterwarnings('ignore', message='.*use_reentrant.*', category=UserWarning)
warnings.filterwarnings('ignore', message='.*GradScaler.*',   category=FutureWarning)

# ── Reset ALL GPUs — clears leftover allocations from previous notebook runs ──
NUM_GPUS = torch.cuda.device_count()
for g in range(NUM_GPUS):
    with torch.cuda.device(g):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(g)

print(f"GPUs detected: {NUM_GPUS}")
for g in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(g)
    free, total = torch.cuda.mem_get_info(g)
    print(f"  GPU {g}: {props.name}  total={total//1024**3} GB  free={free//1024**3} GB")

if NUM_GPUS < 2:
    print("WARNING: Only 1 GPU found — set Accelerator to 'GPU T4 x2' in Kaggle settings.")

# balanced_low_0 → fewer layers on GPU 0 (carries embedding + output head overhead)
DEVICE_MAP     = 'balanced_low_0' if NUM_GPUS > 1 else 'cuda:0'
PRIMARY_DEVICE = torch.device('cuda:0')

print(f"device_map     : {DEVICE_MAP}")
print(f"PRIMARY_DEVICE : {PRIMARY_DEVICE}")

# Warm-up: initialise CUDA context on every GPU
for g in range(NUM_GPUS):
    _ = torch.zeros(1, device=f'cuda:{g}')
    del _
print("CUDA contexts initialised on all GPUs.")


In [ ]:
import gc
import torch
import torch.utils.checkpoint as torch_ckpt
import warnings
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'

# ── Free any model already in memory before reloading ─────────────────────────
try:
    del model
    gc.collect()
    for g in range(torch.cuda.device_count()):
        with torch.cuda.device(g):
            torch.cuda.empty_cache()
    print("Previous model cleared.")
except NameError:
    pass

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


NUM_GPUS = torch.cuda.device_count()

if NUM_GPUS >= 2:
  
    explicit_device_map = {
        'model.visual'          : 0,
        'model.embed_tokens'    : 0,
        'model.rotary_emb'      : 0,
        'lm_head'               : 1,
        'model.norm'            : 1,
    }
    for layer_idx in range(28):
        gpu = 0 if layer_idx < 14 else 1
        explicit_device_map[f'model.layers.{layer_idx}'] = gpu

    LOAD_DEVICE_MAP = explicit_device_map
    print("Using explicit device_map across GPU 0 (layers 0-13) and GPU 1 (layers 14-27)")
else:
    LOAD_DEVICE_MAP = 'cuda:0'
    print("Single GPU mode.")

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map=LOAD_DEVICE_MAP,
    torch_dtype=torch.bfloat16,
)

# ── Verify distribution across GPUs ──────────────────────────────────────────
dev_count = {}
for n, p in model.named_parameters():
    d = str(p.device)
    dev_count[d] = dev_count.get(d, 0) + 1
print("Parameter distribution:")
for dev, cnt in sorted(dev_count.items()):
    print(f"  {dev}: {cnt} parameter tensors")

cpu_params = [n for n, p in model.named_parameters() if p.device.type == 'cpu']
if cpu_params:
    print(f"WARNING: {len(cpu_params)} params on CPU!")
else:
    print("All parameters on GPU ✓")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=128 * 28 * 28,
    max_pixels=256 * 28 * 28,
)

# ── Disable use_cache ─────────────────────────────────────────────────────────
model.config.use_cache = False
if hasattr(model, 'generation_config'):
    model.generation_config.use_cache = False

# ── Freeze visual encoder ─────────────────────────────────────────────────────
for name, param in model.named_parameters():
    if 'visual' in name:
        param.requires_grad_(False)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=False,
)

# ── Kill all gradient checkpointing ──────────────────────────────────────────
def _kill_checkpointing(module):
    for attr in ('gradient_checkpointing', '_gradient_checkpointing',
                 'gradient_checkpointing_func'):
        try:
            if hasattr(module, attr):
                setattr(module, attr,
                        False if attr != 'gradient_checkpointing_func' else None)
        except (AttributeError, TypeError):
            pass

for m in model.modules():
    _kill_checkpointing(m)

try:
    model.model._gradient_checkpointing = False
except AttributeError:
    pass

# ── Monkey-patch checkpoint → passthrough ────────────────────────────────────
def _passthrough_checkpoint(fn, *args, use_reentrant=None, **kwargs):
    return fn(*args, **kwargs)

torch_ckpt.checkpoint = _passthrough_checkpoint
import torch.utils.checkpoint as _tuc
_tuc.checkpoint = _passthrough_checkpoint
print("checkpoint patched → passthrough ✓")

# ── Freeze non-LoRA params ───────────────────────────────────────────────────
for name, param in model.named_parameters():
    if 'lora_' not in name:
        param.requires_grad_(False)

total     = sum(p.numel() for p in model.parameters()) / 1e9
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Total params        : {total:.2f} B")
print(f"Trainable (pre-LoRA): {trainable:.2f} M")

# ── Show per-GPU memory after load ───────────────────────────────────────────
for g in range(torch.cuda.device_count()):
    free, total_mem = torch.cuda.mem_get_info(g)
    used = (total_mem - free) / 1024**3
    print(f"  GPU {g} after model load: {used:.1f}/{total_mem//1024**3} GB used")


In [ ]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=8,                   # ↓ from 16 — halves LoRA activation memory on both GPUs
    lora_alpha=16,         # keep alpha/r ratio = 2
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Re-freeze everything except LoRA after injection
for name, param in model.named_parameters():
    if 'lora_' not in name:
        param.requires_grad_(False)


In [ ]:
from torch.utils.data import Dataset
from PIL import Image, UnidentifiedImageError
import os, torch

INSTRUCTION = 'Convert this document image into clean Markdown format.'

def _is_valid_image(path):
    """Return True only if PIL can open and verify the file."""
    try:
        with Image.open(path) as img:
            img.verify()          
        return True
    except Exception:
        return False


class DocDataset(Dataset):
    def __init__(self, pairs, max_pairs=None):
        candidates = pairs[:max_pairs] if max_pairs else pairs
     
        self.pairs = []
        skipped = 0
        for p in candidates:
            if os.path.getsize(p['image']) > 0 and _is_valid_image(p['image']):
                self.pairs.append(p)
            else:
                skipped += 1
        if skipped:
            print(f"DocDataset: skipped {skipped} corrupt/unreadable images "
                  f"({len(self.pairs)} valid pairs remain)")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        try:
            img = Image.open(p['image']).convert('RGB').resize((512, 512))
        except (UnidentifiedImageError, OSError):
           
            img = Image.new('RGB', (512, 512), color=0)
        with open(p['markdown']) as f:
            md = f.read().strip()
        return img, md


def collate_fn(batch):
    images, markdowns = zip(*batch)
    messages = []
    for img, md in zip(images, markdowns):
        messages.append([
            {'role': 'user',      'content': [{'type': 'image', 'image': img},
                                               {'type': 'text',  'text': INSTRUCTION}]},
            {'role': 'assistant', 'content': [{'type': 'text',  'text': md}]},
        ])

    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in messages
    ]

    from qwen_vl_utils import process_vision_info
    image_inputs, _ = process_vision_info(messages)

    batch_enc = processor(
        text=texts,
        images=image_inputs,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    )

    labels = batch_enc['input_ids'].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch_enc['labels'] = labels

 
    result = {}
    for k, v in batch_enc.items():
        if isinstance(v, torch.Tensor):
            v = v.to(PRIMARY_DEVICE)
            if v.dtype == torch.float32:
                v = v.to(torch.bfloat16)
            result[k] = v
        else:
            result[k] = v
    return result


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 1

train_ds = DocDataset(train_pairs, max_pairs=500)
val_ds   = DocDataset(val_pairs,   max_pairs=100)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False,
)


print("First batch tensor audit:")
_b = next(iter(train_loader))
for k, v in _b.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:30s} device={str(v.device):8s}  dtype={str(v.dtype):20s}  shape={tuple(v.shape)}")
del _b
torch.cuda.empty_cache()

print(f'\nBatch size    : {BATCH_SIZE}')
print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

EPOCHS     = 3
GRAD_ACCUM = 16
LR         = 2e-4
MAX_SEQ    = 512

lora_params = [p for p in model.parameters() if p.requires_grad]
print(f"Optimising {len(lora_params)} LoRA parameter tensors")

optimizer = AdamW(lora_params, lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda')

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for step, batch in enumerate(train_loader):
        # batch already on GPU from collate_fn
        # Safety truncation for any sequence that slipped through
        for k in ('input_ids', 'attention_mask', 'labels'):
            if k in batch and batch[k].shape[-1] > MAX_SEQ:
                batch[k] = batch[k][:, :MAX_SEQ]

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            out  = model(**batch)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        if step % 20 == 0:
            torch.cuda.empty_cache()

    scheduler.step()
    avg_train = total_loss / len(train_loader)
    train_losses.append(avg_train)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            for batch in val_loader:
                for k in ('input_ids', 'attention_mask', 'labels'):
                    if k in batch and batch[k].shape[-1] > MAX_SEQ:
                        batch[k] = batch[k][:, :MAX_SEQ]
                out      = model(**batch)
                val_loss += out.loss.item()
    avg_val = val_loss / len(val_loader)
    val_losses.append(avg_val)

    for g in range(torch.cuda.device_count()):
        free, total_mem = torch.cuda.mem_get_info(g)
        print(f"  GPU {g}: {(total_mem-free)/1024**3:.1f}/{total_mem/1024**3:.1f} GB used")

    elapsed = time.time() - t0
    print(f"Epoch {epoch}/{EPOCHS}  train={avg_train:.4f}  val={avg_val:.4f}  t={elapsed:.1f}s")

print("Training complete.")


In [ ]:
ADAPTER_DIR = os.path.join(OUTPUT_DIR, 'lora_adapter')
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

# Save loss history
with open(os.path.join(OUTPUT_DIR, 'loss_history.json'), 'w') as f:
    json.dump({'train': train_losses, 'val': val_losses}, f)

print('Adapter saved to', ADAPTER_DIR)